In [1]:
import cv2
import numpy as np
import time
from collections import deque

import ipywidgets as widgets
import traitlets
from IPython.display import display

from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

# Nếu đã có camera từ notebook khác, không tạo lại nhiều lần
try:
    camera.running = False
except:
    pass

camera = CSICamera(width=224, height=224)
camera.running = True

camera_widget = widgets.Image(format='jpeg', width=224, height=224)

camera_link = traitlets.dlink(
    (camera, 'value'),
    (camera_widget, 'value'),
    transform=bgr8_to_jpeg
)

display(camera_widget)

print("Camera started")

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Camera started


In [2]:
class LaneDetector:
    def __init__(self):
        self.roi_start_ratio = 0.35

        # ROI hình thang, chỉnh theo camera
        self.poly_top_left = 35
        self.poly_top_right = 190
        self.poly_bottom_left = 5
        self.poly_bottom_right = 219

        # Threshold vạch trắng
        # Nếu bắt thiếu vạch: giảm V từ 145 xuống 125
        # Nếu bắt quá nhiều nền trắng: tăng V lên 165 và giảm S upper
        self.lower_white = np.array([0, 0, 145])
        self.upper_white = np.array([180, 80, 255])

        # Threshold mặt đường đen để loại nền trắng ngoài đường
        self.lower_black = np.array([0, 0, 0])
        self.upper_black = np.array([180, 255, 115])

        # Lane width fallback nếu chỉ thấy một vạch
        self.lane_width_pixels = 90

        # Chọn lane:
        # "RIGHT_LANE": chạy giữa vạch đứt bên trái và vạch liền bên phải
        # "LEFT_LANE": chạy giữa vạch liền bên trái và vạch đứt bên phải
        self.target_lane = "RIGHT_LANE"

        self.lookahead_ratio = 0.75
        self.error_history = deque(maxlen=2)

        self.last_center = None
        self.lost_count = 0
        self.max_lost_count = 5

        self.color_mode = "BGR"

    def apply_roi(self, mask):
        h, w = mask.shape[:2]
        roi_mask = np.zeros_like(mask)

        polygon = np.array([[
            (self.poly_top_left, 0),
            (self.poly_top_right, 0),
            (self.poly_bottom_right, h - 1),
            (self.poly_bottom_left, h - 1)
        ]], dtype=np.int32)

        cv2.fillPoly(roi_mask, polygon, 255)
        return cv2.bitwise_and(mask, roi_mask)

    def fit_x_from_y(self, points):
        if len(points) < 6:
            return None

        xs = np.array([p[0] for p in points], dtype=np.float32)
        ys = np.array([p[1] for p in points], dtype=np.float32)

        a, b = np.polyfit(ys, xs, 1)
        return a, b

    def x_at_y(self, line, y):
        if line is None:
            return None
        a, b = line
        return int(a * y + b)

    def get_white_lane_mask(self, roi):
        if self.color_mode == "BGR":
            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        else:
            hsv = cv2.cvtColor(roi, cv2.COLOR_RGB2HSV)

        white_mask = cv2.inRange(hsv, self.lower_white, self.upper_white)
        black_mask = cv2.inRange(hsv, self.lower_black, self.upper_black)

        # Chỉ xét trong ROI hình thang
        white_mask = self.apply_roi(white_mask)
        black_mask = self.apply_roi(black_mask)

        # Mở rộng vùng mặt đường đen một chút để giữ vạch trắng nằm sát mặt đường
        road_kernel = np.ones((17, 17), np.uint8)
        road_area = cv2.dilate(black_mask, road_kernel, iterations=1)

        # Chỉ giữ vạch trắng nằm gần/thuộc vùng mặt đường
        lane_mask = cv2.bitwise_and(white_mask, road_area)

        # Làm sạch nhiễu nhỏ
        lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

        # Nối nhẹ vạch đứt theo chiều dọc
        lane_mask = cv2.morphologyEx(lane_mask, cv2.MORPH_CLOSE, np.ones((13, 3), np.uint8))

        return lane_mask

    def extract_lane_lines(self, lane_mask):
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(lane_mask, connectivity=8)

        lines = []

        for i in range(1, num_labels):
            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            area = stats[i, cv2.CC_STAT_AREA]

            # Bỏ nhiễu quá nhỏ
            if area < 20:
                continue

            # Bỏ mảng trắng quá rộng, thường là nền/tường
            if w > 55 and h < 35:
                continue

            # Lấy points của component
            ys, xs = np.where(labels == i)

            points = list(zip(xs, ys))
            line = self.fit_x_from_y(points)

            if line is None:
                continue

            # Vị trí x tại lookahead để phân loại trái/phải
            lookahead_y = int(lane_mask.shape[0] * self.lookahead_ratio)
            x_at_lookahead = self.x_at_y(line, lookahead_y)

            if x_at_lookahead is None:
                continue

            lines.append({
                "line": line,
                "x": x_at_lookahead,
                "area": area,
                "bbox": (x, y, w, h)
            })

        lines = sorted(lines, key=lambda item: item["x"])
        return lines

    def select_lane_boundaries(self, lines, image_center):
        """
        Chọn 2 vạch trắng làm biên lane.
        RIGHT_LANE:
            left boundary = vạch trắng gần bên trái center
            right boundary = vạch trắng bên phải
        LEFT_LANE:
            left boundary = vạch bên trái
            right boundary = vạch gần bên phải center
        """
        if len(lines) < 1:
            return None, None

        left_lines = [ln for ln in lines if ln["x"] < image_center]
        right_lines = [ln for ln in lines if ln["x"] >= image_center]

        left_boundary = None
        right_boundary = None

        if self.target_lane == "RIGHT_LANE":
            # Vạch đứt thường nằm gần center/trái, vạch liền nằm phải
            if len(left_lines) > 0:
                left_boundary = max(left_lines, key=lambda ln: ln["x"])
            if len(right_lines) > 0:
                right_boundary = min(right_lines, key=lambda ln: ln["x"])

        else:
            # LEFT_LANE
            if len(left_lines) > 0:
                left_boundary = max(left_lines, key=lambda ln: ln["x"])
            if len(right_lines) > 0:
                right_boundary = min(right_lines, key=lambda ln: ln["x"])

        left_line = None if left_boundary is None else left_boundary["line"]
        right_line = None if right_boundary is None else right_boundary["line"]

        return left_line, right_line

    def detect(self, image):
        h, w = image.shape[:2]

        roi_y = int(h * self.roi_start_ratio)
        roi = image[roi_y:h, :]
        roi_h = roi.shape[0]

        lane_mask = self.get_white_lane_mask(roi)
        lines = self.extract_lane_lines(lane_mask)

        image_center = w // 2
        lookahead_y = int(roi_h * self.lookahead_ratio)

        left_line, right_line = self.select_lane_boundaries(lines, image_center)

        left_x = self.x_at_y(left_line, lookahead_y)
        right_x = self.x_at_y(right_line, lookahead_y)

        lane_center = None
        center_line = None

        if left_line is not None and right_line is not None:
            left_x = self.x_at_y(left_line, lookahead_y)
            right_x = self.x_at_y(right_line, lookahead_y)

            lane_center = int((left_x + right_x) / 2)

            a1, b1 = left_line
            a2, b2 = right_line
            center_line = ((a1 + a2) / 2, (b1 + b2) / 2)

        elif left_line is not None:
            left_x = self.x_at_y(left_line, lookahead_y)
            lane_center = int(left_x + self.lane_width_pixels / 2)

            a, b = left_line
            center_line = (a, b + self.lane_width_pixels / 2)

        elif right_line is not None:
            right_x = self.x_at_y(right_line, lookahead_y)
            lane_center = int(right_x - self.lane_width_pixels / 2)

            a, b = right_line
            center_line = (a, b - self.lane_width_pixels / 2)

        else:
            if self.last_center is not None and self.lost_count < self.max_lost_count:
                lane_center = self.last_center
                self.lost_count += 1
            else:
                return {
                    "ok": False,
                    "error": None,
                    "lane_center": None,
                    "left_x": None,
                    "right_x": None,
                    "mask": lane_mask,
                    "roi_y": roi_y,
                    "left_line": None,
                    "right_line": None,
                    "center_line": None,
                    "lookahead_y": lookahead_y,
                    "lines": lines
                }

        lane_center = max(0, min(w - 1, lane_center))

        self.last_center = lane_center
        self.lost_count = 0

        error = lane_center - image_center

        self.error_history.append(error)
        smooth_error = sum(self.error_history) / len(self.error_history)

        return {
            "ok": True,
            "error": smooth_error,
            "lane_center": lane_center,
            "left_x": left_x,
            "right_x": right_x,
            "mask": lane_mask,
            "roi_y": roi_y,
            "left_line": left_line,
            "right_line": right_line,
            "center_line": center_line,
            "lookahead_y": lookahead_y,
            "lines": lines
        }


detector = LaneDetector()
print("White lane marking detector ready")

White lane marking detector ready


In [9]:
import math
import time

def get_road_geometry(result, detector, image):
    """
    Tính geometry điều khiển:
    - angle_deg: góc lệch giữa hướng lane và trục camera
    - lateral_error: target lane lệch so với camera center

    Bản này có thêm:
    - TARGET_OFFSET_PX để né line viền
    - SINGLE_LINE_EXTRA_OFFSET_PX để không bám 1 vạch duy nhất
    - border guard để target không nằm quá sát line viền
    """

    h, w = image.shape[:2]

    roi_y = result.get("roi_y", int(h * detector.roi_start_ratio))
    roi_h = h - roi_y

    center_line = result.get("center_line", None)

    # Điểm xa/gần để tính góc
    y_far = int(roi_h * 0.35)
    y_near = int(roi_h * 0.82)

    if center_line is not None:
        a, b = center_line
        x_far = int(a * y_far + b)
        x_near = int(a * y_near + b)
    else:
        x_near = result.get("lane_center", w // 2)
        if x_near is None:
            x_near = w // 2
        x_far = x_near

    left_x = result.get("left_x")
    right_x = result.get("right_x")

    # ==================================================
    # CHỈNH Ở ĐÂY
    # Nếu xe bám line viền phải: dùng số âm.
    # Nếu xe bám line viền trái: đổi thành số dương.
    # ==================================================
    TARGET_OFFSET_PX = -18

    # Khi chỉ thấy 1 vạch, đẩy target ra xa vạch đó hơn
    SINGLE_LINE_EXTRA_OFFSET_PX = 30

    # Không cho target quá sát line viền phải/trái
    MIN_RIGHT_CLEARANCE_PX = 48
    MIN_LEFT_CLEARANCE_PX = 34

    border_risk = False

    # Offset mặc định để né viền phải
    x_near += TARGET_OFFSET_PX
    x_far += int(TARGET_OFFSET_PX * 0.7)

    # Nếu chỉ thấy line phải, rất dễ bám line viền phải -> đẩy target sang trái thêm
    if left_x is None and right_x is not None:
        x_near -= SINGLE_LINE_EXTRA_OFFSET_PX
        x_far -= int(SINGLE_LINE_EXTRA_OFFSET_PX * 0.7)
        border_risk = True

    # Nếu chỉ thấy line trái -> đẩy target sang phải thêm
    elif right_x is None and left_x is not None:
        x_near += SINGLE_LINE_EXTRA_OFFSET_PX
        x_far += int(SINGLE_LINE_EXTRA_OFFSET_PX * 0.7)
        border_risk = True

    # Guard line phải: nếu target gần line phải quá thì kéo target sang trái
    if right_x is not None:
        dist_right = right_x - x_near

        if 0 < dist_right < MIN_RIGHT_CLEARANCE_PX:
            shift = MIN_RIGHT_CLEARANCE_PX - dist_right
            x_near -= shift
            x_far -= int(shift * 0.7)
            border_risk = True

    # Guard line trái: nếu target gần line trái quá thì kéo target sang phải
    if left_x is not None:
        dist_left = x_near - left_x

        if 0 < dist_left < MIN_LEFT_CLEARANCE_PX:
            shift = MIN_LEFT_CLEARANCE_PX - dist_left
            x_near += shift
            x_far += int(shift * 0.7)
            border_risk = True

    # Giới hạn trong ảnh
    x_near = clamp(x_near, 0, w - 1)
    x_far = clamp(x_far, 0, w - 1)

    image_center = w // 2

    lateral_error = x_near - image_center

    dx = x_near - x_far
    dy = y_near - y_far

    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)

    return {
        "x_far": int(x_far),
        "y_far": int(roi_y + y_far),
        "x_near": int(x_near),
        "y_near": int(roi_y + y_near),
        "lateral_error": lateral_error,
        "angle_deg": angle_deg,
        "border_risk": border_risk
    }

In [3]:
debug_widget = widgets.Image(format='jpeg', width=448, height=224)
status_widget = widgets.HTML(value="Debug status: idle")

display(debug_widget)
display(status_widget)

debug_running = False

def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def draw_fitted_line(debug, line, roi_y, roi_h, color, thickness=2):
    if line is None:
        return

    a, b = line

    y1_roi = int(roi_h * 0.20)
    y2_roi = roi_h - 1

    x1 = int(a * y1_roi + b)
    x2 = int(a * y2_roi + b)

    y1 = roi_y + y1_roi
    y2 = roi_y + y2_roi

    cv2.line(debug, (x1, y1), (x2, y2), color, thickness)


def make_debug_image(image, result):
    h, w = image.shape[:2]
    debug = image.copy()

    roi_y = int(h * detector.roi_start_ratio)
    roi_h = h - roi_y

    # ROI hình thang
    polygon = np.array([[
        (detector.poly_top_left, roi_y),
        (detector.poly_top_right, roi_y),
        (detector.poly_bottom_right, h - 1),
        (detector.poly_bottom_left, h - 1)
    ]], dtype=np.int32)

    cv2.polylines(debug, polygon, True, (0, 165, 255), 2)

    # Trục giữa camera
    cv2.line(debug, (w // 2, 0), (w // 2, h), (255, 255, 0), 2)

    # Vẽ tất cả line detect được bằng màu xám nhạt
    for item in result.get("lines", []):
        draw_fitted_line(debug, item["line"], roi_y, roi_h, (180, 180, 180), 1)

    # Biên lane đã chọn
    draw_fitted_line(debug, result.get("left_line"), roi_y, roi_h, (255, 0, 0), 2)
    draw_fitted_line(debug, result.get("right_line"), roi_y, roi_h, (0, 255, 0), 2)

    # Center lane
    draw_fitted_line(debug, result.get("center_line"), roi_y, roi_h, (0, 0, 255), 3)

    # Điểm lookahead
    if result.get("lane_center") is not None:
        lookahead_y_img = roi_y + result["lookahead_y"]
        cv2.circle(debug, (result["lane_center"], lookahead_y_img), 5, (0, 0, 255), -1)
        cv2.line(debug, (0, lookahead_y_img), (w - 1, lookahead_y_img), (255, 255, 255), 1)

    # Hiện thông tin góc nếu có
    try:
        geom = get_road_geometry(result, detector, image)
        text = "angle: {:.1f} deg | lat: {:.1f}px".format(
            geom["angle_deg"],
            geom["lateral_error"]
        )

        cv2.putText(
            debug,
            text,
            (5, 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    except:
        pass

    # Mask bên phải
    mask = result["mask"]
    mask_color = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    mask_color = cv2.resize(mask_color, (w, h))

    combined = np.concatenate([debug, mask_color], axis=1)
    return combined

Image(value=b'', format='jpeg', height='224', width='448')

HTML(value='Debug status: idle')

In [11]:
def live_debug(duration_sec=10):
    start = time.time()
    
    while time.time() - start < duration_sec:
        image = camera.value
        result = detector.detect(image)
        debug_img = make_debug_image(image, result)
        debug_widget.value = bgr8_to_jpeg(debug_img)
        
        print(
            "ok:", result["ok"],
            "error:", None if result["error"] is None else round(result["error"], 2),
            "left:", result["left_x"],
            "right:", result["right_x"]
        )
        
        time.sleep(0.15)

live_debug(duration_sec=10)

ok: True error: 39.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 right: 147
ok: True error: -31.0 left: 15 righ

In [5]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()
car.throttle = 0.0
car.steering = 0.0

print("Motor stopped")

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Motor stopped


In [10]:
def clamp(x, lo, hi):
    return max(min(x, hi), lo)


def apply_steering_smoothing(target, previous, alpha=0.72, max_delta=0.16):
    smoothed = previous + alpha * (target - previous)

    delta = smoothed - previous
    delta = clamp(delta, -max_delta, max_delta)

    return previous + delta


def enforce_min_abs_steering(steering, min_abs, fallback_sign=1):
    if abs(steering) >= min_abs:
        return steering

    if steering > 0:
        return min_abs
    elif steering < 0:
        return -min_abs
    else:
        return min_abs if fallback_sign >= 0 else -min_abs


def run_lane_following(duration_sec=18):
    # Xe của bạn đang cần đảo dấu steering
    STEERING_SIGN = -1

    # =========================
    # GAIN
    # =========================
    K_angle_normal = 0.026
    K_lat_normal = 0.0035

    K_angle_curve = 0.060
    K_lat_curve = 0.0060

    K_angle_recovery = 0.085
    K_lat_recovery = 0.0090

    steering_bias = 0.0

    # =========================
    # THROTTLE: giữ theo bản bạn đang chạy được
    # =========================
    throttle_normal = 0.15
    throttle_curve = 0.12
    throttle_recovery = 0.10
    throttle_lost = 0.020

    # =========================
    # STEERING LIMIT
    # =========================
    max_steering_normal = 0.50
    max_steering_curve = 0.85
    max_steering_recovery = 1.00

    min_recovery_steering = 0.65

    # =========================
    # MODE THRESHOLDS
    # =========================
    curve_angle_enter = 5.0
    curve_angle_exit = 3.0

    recovery_lat_enter = 35.0
    recovery_lat_exit = 22.0

    recovery_angle_enter = 18.0
    recovery_angle_exit = 10.0

    # =========================
    # SMOOTHING
    # =========================
    steering_alpha = 0.72

    max_steering_delta_normal = 0.11
    max_steering_delta_curve = 0.18
    max_steering_delta_recovery = 0.25

    debug_every_n_frames = 2
    print_every_n_frames = 3

    lost_frames = 0
    max_lost_frames = 4

    border_risk_frames = 0
    max_border_risk_frames = 10

    last_steering = 0.0
    last_mode = "NORMAL"
    frame_id = 0

    try:
        detector.error_history.clear()
    except:
        pass

    car.throttle = 0.0
    car.steering = 0.0

    print("Starting in 2 seconds...")
    time.sleep(2)

    start = time.time()

    try:
        while time.time() - start < duration_sec:
            frame_id += 1

            image = camera.value
            result = detector.detect(image)

            # =========================
            # LOST LANE HANDLING
            # =========================
            if (not result["ok"]) or result.get("lane_center") is None:
                lost_frames += 1

                if lost_frames <= max_lost_frames:
                    safe_steering = clamp(last_steering, -0.75, 0.75)

                    car.steering = safe_steering
                    car.throttle = throttle_lost

                    if frame_id % print_every_n_frames == 0:
                        print(
                            "LOST_RECOVERY",
                            "lost:", lost_frames,
                            "steering:", round(safe_steering, 3),
                            "throttle:", throttle_lost
                        )

                    time.sleep(0.04)
                    continue

                car.throttle = 0.0
                car.steering = 0.0
                print("STOP: lane lost too long")
                break

            lost_frames = 0

            # =========================
            # ROAD GEOMETRY
            # =========================
            geom = get_road_geometry(result, detector, image)

            angle_error = geom["angle_deg"]
            lateral_error = geom["lateral_error"]
            border_risk = geom.get("border_risk", False)

            if border_risk:
                border_risk_frames += 1
            else:
                border_risk_frames = 0

            abs_angle = abs(angle_error)
            abs_lat = abs(lateral_error)

            # Nếu bám viền quá lâu thì coi như recovery
            force_recovery = border_risk_frames >= max_border_risk_frames

            # =========================
            # MODE SELECTION WITH HYSTERESIS
            # =========================
            if force_recovery:
                mode = "RECOVERY"

            elif last_mode == "RECOVERY":
                if abs_lat <= recovery_lat_exit and abs_angle <= recovery_angle_exit:
                    if abs_angle >= curve_angle_enter:
                        mode = "CURVE"
                    else:
                        mode = "NORMAL"
                else:
                    mode = "RECOVERY"

            elif last_mode == "CURVE":
                if abs_lat >= recovery_lat_enter or abs_angle >= recovery_angle_enter:
                    mode = "RECOVERY"
                elif abs_angle <= curve_angle_exit:
                    mode = "NORMAL"
                else:
                    mode = "CURVE"

            else:
                if abs_lat >= recovery_lat_enter or abs_angle >= recovery_angle_enter:
                    mode = "RECOVERY"
                elif abs_angle >= curve_angle_enter or border_risk:
                    mode = "CURVE"
                else:
                    mode = "NORMAL"

            last_mode = mode

            # =========================
            # PARAMS BY MODE
            # =========================
            if mode == "RECOVERY":
                K_angle = K_angle_recovery
                K_lat = K_lat_recovery

                throttle = throttle_recovery
                max_steering = max_steering_recovery
                max_delta = max_steering_delta_recovery

            elif mode == "CURVE":
                K_angle = K_angle_curve
                K_lat = K_lat_curve

                throttle = throttle_curve
                max_steering = max_steering_curve
                max_delta = max_steering_delta_curve

            else:
                K_angle = K_angle_normal
                K_lat = K_lat_normal

                throttle = throttle_normal
                max_steering = max_steering_normal
                max_delta = max_steering_delta_normal

            # =========================
            # STEERING CONTROL
            # =========================
            raw_control = K_angle * angle_error + K_lat * lateral_error
            target_steering = STEERING_SIGN * raw_control + steering_bias

            target_steering = clamp(target_steering, -max_steering, max_steering)

            if mode == "RECOVERY":
                fallback_sign = 1 if target_steering >= 0 else -1

                target_steering = enforce_min_abs_steering(
                    target_steering,
                    min_recovery_steering,
                    fallback_sign=fallback_sign
                )

                target_steering = clamp(target_steering, -max_steering, max_steering)

            steering = apply_steering_smoothing(
                target=target_steering,
                previous=last_steering,
                alpha=steering_alpha,
                max_delta=max_delta
            )

            steering = clamp(steering, -max_steering, max_steering)

            car.steering = steering
            car.throttle = throttle

            last_steering = steering

            # =========================
            # DEBUG VIEW
            # =========================
            if frame_id % debug_every_n_frames == 0:
                try:
                    debug_img = make_debug_image(image, result)
                    debug_widget.value = bgr8_to_jpeg(debug_img)
                except:
                    pass

            if frame_id % print_every_n_frames == 0:
                print(
                    mode,
                    "angle:", round(angle_error, 2),
                    "lat:", round(lateral_error, 2),
                    "border:", border_risk,
                    "border_frames:", border_risk_frames,
                    "raw:", round(raw_control, 3),
                    "target:", round(target_steering, 3),
                    "steering:", round(steering, 3),
                    "throttle:", throttle
                )

            time.sleep(0.04)

    except KeyboardInterrupt:
        print("Interrupted")

    finally:
        car.throttle = 0.0
        car.steering = 0.0
        print("STOPPED")


run_lane_following(duration_sec=18)

Starting in 2 seconds...
CURVE angle: 14.04 lat: 25 border: True border_frames: 3 raw: 0.992 target: -0.85 steering: -0.54 throttle: 0.12
CURVE angle: 14.04 lat: 25 border: True border_frames: 6 raw: 0.992 target: -0.85 steering: -0.84 throttle: 0.12
CURVE angle: 8.37 lat: 3 border: False border_frames: 0 raw: 0.52 target: -0.52 steering: -0.669 throttle: 0.12
NORMAL angle: 0.0 lat: -12 border: False border_frames: 0 raw: -0.042 target: 0.042 steering: -0.232 throttle: 0.15
NORMAL angle: -1.68 lat: -14 border: False border_frames: 0 raw: -0.093 target: 0.093 steering: 0.063 throttle: 0.15
NORMAL angle: -2.53 lat: -15 border: False border_frames: 0 raw: -0.118 target: 0.118 steering: 0.117 throttle: 0.15
NORMAL angle: -2.53 lat: -14 border: False border_frames: 0 raw: -0.115 target: 0.115 steering: 0.089 throttle: 0.15
NORMAL angle: 0.0 lat: -10 border: False border_frames: 0 raw: -0.035 target: 0.035 steering: 0.042 throttle: 0.15
NORMAL angle: 0.0 lat: -12 border: False border_frames: